# Anonimizacja danych osobowych w tekście

Notebook pokazuje, jak za pomocą Presidio wykrywać i anonimizować dane osobowe w polskim tekście. 
Najpierw konfiguruje środowisko z polskim modelem spaCy, potem rozpoznaje encje takie jak osoba, tytuł, zaimek i numer telefonu, a na końcu anonimizuje wskazane dane w tekście wejściowym.

In [ ]:
# pobranie biblioteki Presidio
%pip install spacy
%pip install presidio_analyzer presidio_anonymizer
!python -m spacy download pl_core_news_sm

from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import SpacyNlpEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import json
from pprint import pprint

# Analiza tekstu pod kątem encji PII

Za pomocą Presidio Analyzer przeanalizuj tekst, aby zidentyfikować encje PII.
Analizator Presidio korzysta z predefiniowanych rozpoznawaczy encji i umożliwia tworzenie własnych rozpoznawaczy.

Poniższy przykład kodu:

- Konfiguruje silnik Analyzer: wczytuje moduł NLP (domyślnie model spaCy) oraz inne rozpoznawacze PII
- Wywołuje analizator, aby uzyskać wyniki dla encji typu "PHONE_NUMBER"


In [ ]:
text_to_anonymize = "Nazywa się Pan Kowalski, a jego numer telefonu to +48 123 456 789"

In [ ]:
nlp_engine = SpacyNlpEngine(models=[{"lang_code": "pl", "model_name": "pl_core_news_sm"}])
nlp_engine.load()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)
person_recognizer = PatternRecognizer(supported_entity="PERSON",
                                     supported_language="pl",
                                     deny_list=["Kowalski"])
analyzer.registry.add_recognizer(person_recognizer)
phone_number_recognizer = PatternRecognizer(supported_entity="PHONE_NUMBER",
                                            supported_language="pl",
                                            patterns=[Pattern(name="polish_phone_number",
                                                              regex=r"\b(?:\+48[\s-]?)?(?:\d[\s-]?){9}\b",
                                                              score=0.85)])
analyzer.registry.add_recognizer(phone_number_recognizer)
analyzer_results = analyzer.analyze(text=text_to_anonymize, entities=["PHONE_NUMBER"], language='pl')

print(analyzer_results)

## Tworzenie własnych rozpoznawaczy encji PII

Presidio Analyzer zawiera predefiniowany zestaw rozpoznawaczy encji. Umożliwia także dodawanie nowych rozpoznawaczy bez zmiany bazowego kodu analizatora, **tworząc własne rozpoznawacze**.
W poniższym przykładzie utworzymy dwa nowe rozpoznawacze typu `PatternRecognizer`, aby identyfikować tytuły i zaimki w analizowanym tekście.
`PatternRecognizer` to rozpoznawacz encji PII, który korzysta z wyrażeń regularnych lub listy wykluczeń.

Poniższy przykład kodu:
- Tworzy własne rozpoznawacze
- Dodaje nowe własne rozpoznawacze do analizatora
- Wywołuje analizator, aby uzyskać wyniki z nowych rozpoznawaczy

In [ ]:
titles_recognizer = PatternRecognizer(supported_entity="TITLE",
                                      supported_language="pl",
                                      deny_list=["Pan","Pani","Panna"])

pronoun_recognizer = PatternRecognizer(supported_entity="PRONOUN",
                                       supported_language="pl",
                                       deny_list=["on", "On", "jego", "Jego", "ona", "Ona", "jej", "Jej"])

analyzer.registry.add_recognizer(titles_recognizer)
analyzer.registry.add_recognizer(pronoun_recognizer)

analyzer_results = analyzer.analyze(text=text_to_anonymize,
                            entities=["TITLE", "PRONOUN"],
                            language="pl")
print(analyzer_results)


Wywołaj Presidio Analyzer i pobierz przeanalizowane wyniki ze wszystkimi skonfigurowanymi rozpoznawaczami - domyślnymi i nowymi własnymi

In [ ]:
analyzer_results = analyzer.analyze(text=text_to_anonymize, language='pl')

analyzer_results

# Anonimizacja tekstu z wykrytymi encjami PII

<br>Presidio Anonymizer przechodzi po wynikach Presidio Analyzer i zapewnia możliwości anonimizacji wykrytego tekstu.
<br>Anonymizer udostępnia 5 typów operacji anonimizacji - replace, redact, mask, hash i encrypt. Domyślnie używany jest **replace**

<br>Poniższy przykład kodu:
<ol>
<li>Konfiguruje silnik anonimizacji </li>
<li>Tworzy żądanie anonimizacji - tekst do anonimizacji, listę operacji anonimizacji do zastosowania oraz wyniki z analizatora</li>
<li>Anonimizuje tekst</li>
</ol>

In [ ]:
anonymizer = AnonymizerEngine()

pii_results = [result for result in analyzer_results if result.entity_type in {"TITLE", "PERSON", "PHONE_NUMBER"}]

anonymized_results = anonymizer.anonymize(
    text=text_to_anonymize,
    analyzer_results=pii_results,    
    operators={"PERSON": OperatorConfig("replace", {"new_value": "<ANONYMIZED>"}),
                "PHONE_NUMBER": OperatorConfig("mask", {"type": "mask", "masking_char" : "*", "chars_to_mask" : 12, "from_end" : True})}
)

print(f"tekst: {anonymized_results.text}")
print("szczegółowa odpowiedź:")

pprint(json.loads(anonymized_results.to_json()))